## [Langchain Basic RAG](https://docs.langchain.com/oss/python/langchain/knowledge-base#search-by-vector)

### Pipeline overview

`load -> split -> embed -> store -> retrieve -> generate`

This notebook builds every stage by hand once, so later notebooks can swap out individual stages.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]="Rag-basic"

### Task 1 - Documents

- `Document` = LangChain's universal unit of text: `page_content` + `metadata` dict.
- Every loader/splitter/store speaks this format, so pieces stay interchangeable.

In [ ]:
# Create documents

from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

### Task 2 - Embeddings

- `embed_query(text)` -> one fixed-length float vector per text (here: 768-dim via nomic-embed-text).
- Similar meanings land close together in vector space - that is all retrieval relies on.
- Runs locally through Ollama, no API key needed.

In [ ]:
# Generate embeddings
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='nomic-embed-text')

Inspecting the output: same length for any input, values are just floats.
You will never handle these vectors manually again - stores do it for you.

In [ ]:
vector1 = embeddings.embed_query(documents[0].page_content)
vector2 = embeddings.embed_query(documents[1].page_content)

assert len(vector1) == len(vector2)
print(f"Generated vectors of length {len(vector1)}\n")
vector1[:10]

### Task 3 - Vector store

- `InMemoryVectorStore` = vectors + metadata in RAM. Dies with the kernel.
- Persistent alternatives used later: Chroma (`rag_indexing`, `rag_query_translation`).
- One store, two roles: `add_documents` (write) and `similarity_search` (read).

In [ ]:
# Vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### Task 4 - Load a real PDF

- Each PDF page becomes one `Document`; page number kept in metadata (useful for citations).
`extract_text()` returns `None` on image-only pages - hence the `or ""`.

In [ ]:
# Load a PDF
from langchain_core.documents import Document
import pypdf

def load_pdf(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page":i},
        )
        for i, page in enumerate(reader.pages)
    ]

import os

# Resolve corpus files whether the kernel was launched from the repo root or rag_playbooks/
def data_path(name: str) -> str:
    for base in (os.getcwd(), os.path.dirname(os.getcwd())):
        p = os.path.join(base, "rag_input_docs", name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"rag_input_docs/{name} not found - place your source docs there")

docs = load_pdf(data_path("SQLAlchemy_with_Postgres.pdf"))
print(len(docs))
docs

### Task 5 - Splitting

- Whole pages are too big for embeddings AND for LLM context -> chunk them.
- `RecursiveCharacterTextSplitter`: tries paragraph -> sentence -> word separators until chunks fit `chunk_size`.
- `chunk_overlap=200`: repeats ~200 chars between neighbors so sentences cut in half still appear fully somewhere.
- `add_start_index=True`: stores char offset in metadata - lets you locate a chunk inside its source page.
- Deep dive: `rag_chunking` notebook.

In [ ]:
# Split a Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

len(all_splits)
# all_splits

### Task 6 - Indexing (batched)

- Embedding calls have real cost/latency -> upload in batches of 20 instead of all at once.
- `add_documents` embeds each chunk and stores `(vector, text, metadata)`; returns generated IDs.

In [ ]:
# Index Documents
for i in range(0, len(all_splits), 20):
    batch = all_splits[i:i+20]
    ids = vector_store.add_documents(documents=batch)
    print(ids)

### Task 7 - Four ways to search

The next four cells show the same retrieval from different angles:

| Cell | What it demonstrates |
|---|---|
| `similarity_search` | query string -> top-k Documents |
| `asimilarity_search` | same, async (matters when serving many requests) |
| `similarity_search_with_score` | also returns the distance - lower = more similar here |
| `similarity_search_by_vector` | skip the string: search directly with a pre-computed embedding |

In [ ]:
# Search by string
results = vector_store.similarity_search(
    "Whats the query Lifecycle?"
)

print(results[0])

In [ ]:
# Async query:

results = await vector_store.asimilarity_search("Diff between core vs orm?")
print(results[0])

In [ ]:
# Return Scores
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("Whats the query Lifecycle?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

In [ ]:
# Search by vector
embedding = embeddings.embed_query("Diff between core vs orm?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

### Task 8 - Retriever = Runnable wrapper around search

- Key distinction: **VectorStore is NOT a Runnable** - you cannot pipe it into chains or call `.batch()`.
- A **Retriever IS a Runnable**: takes a plain string, returns Documents.
- `@chain` turns any function into a Runnable - here we hand-roll a retriever with k=3.

`.batch(queries)` runs all 4 queries through the same function - this is why everything
in LangChain wants to be a Runnable.

In [ ]:
# Use retrievers

# LangChain VectorStore objects do not subclass Runnable. 
# Retrievers are Runnables, so they support standard methods such as sync and async invoke and batch.

from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retreiver(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=3)

queries = [
    "Diff between core vs orm?",
    "What's the query Lifecycle?",
    "What's Mapped and Mapped Column?",
    "Session vs Scalar"
]

retreiver.batch(queries)


Alternative (commented out): every store offers `.as_retriever(...)` which does the same
without writing the function yourself. Hand-rolling wins when you need custom logic inside.

In [ ]:
# vector Store retreivers

# retriever = vector_store.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 1},
# )

# retriever.batch(queries)


### Task 9 - The full RAG chain

Steps inside `rag_chain`:

1. `retreiver.invoke(question)` - fetch top-3 chunks (Task 8)
2. join chunk texts into one context string
3. system prompt: instructions + context wrapped in `<context>` tags
4. user message: the raw question
5. return answer + retrieved docs + raw message (docs are handy for debugging/citations)

> `@traceable` logs this whole function as one LangSmith trace - open the project link
> to see retrieval and generation as nested steps.

In [ ]:
from langchain_ollama import ChatOllama
from langsmith import traceable

llm = ChatOllama(model='gemma3:4b', temperature=0.5)

@traceable
def rag_chain(question: str) -> dict:
    docs = retreiver.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
       Use the following source documents to answer the user's questions.
       If you don't know the answer, just say that you don't know.
       Use three sentences maximum and keep the answer concise.

<context>
{docs_string}
</context>"""

    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )

    return {"answer": ai_msg.content, "documents":docs, "metadata":ai_msg}

In [ ]:
question = "Diff between core vs orm? Also can you provide code example for both"

ans = rag_chain(question)

ans["answer"]